# ARM KAZR Cloud Persistence Project: Annual Target Sketch

**Alec Blevins — STA 496, Fall 2026**

## Purpose
We will reconstruct `06_target_sketch.ipynb` over the full 2024 year to choose target and horizon. 

The state series will be built once across all days so midnight is handled. The binning and target code here will motivate `aggregation.py` and `targets.py`. 

This notebook is meant to be less exploratory, and more confirm decisions for future modules. 

## 1. Imports & Paths

In [1]:
from pathlib import Path

import sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import display

#Resolve project root and set up paths
_start = Path.cwd().resolve()
PROJECT_ROOT = next(c for c in [_start, *_start.parents] if (c/"ARM_DATA"/"2024-2025").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

#Import project-specific modules
from src.qc import apply_qc
from src.cloud_state import construct_cloud_state, concat_cloud_state_frames

#HERE, we will use the interim data files for the 2024-2025 period. Then apply QC in memory. 
INTERIM = PROJECT_ROOT/"ARM_DATA"/"2024-2025"/"interim"/"sgparsclkazr1kolliasC1.c0"
files = sorted(f for f in INTERIM.glob("kazr_2024*.nc"))
print(len(files), "files")  # expect 366

366 files


## 2. Apply QC

In [2]:
#Process each file, apply QC, construct cloud state, and concatenate results
frames = []

for f in files:
    with xr.open_dataset(f) as ds:
        qc = apply_qc(ds) #Apply QC
        fr = construct_cloud_state(qc)[["cloud_state"]] #Construct cloud state
    frames.append(fr) #Append the processed frame to the list
    
state4 = concat_cloud_state_frames(frames)["cloud_state"] #Concatenate all processed frames
assert len(state4) == 366*21600 #Check that the total length matches expectation
print(state4.value_counts().rename({1:"Cloudy",0:"Clear",-1:"Unknown"}))  # unknown ≈ 13,474

cloud_state
Clear      4069341
Cloudy     3822785
Unknown      13474
Name: count, dtype: int64


## 3. Bin Construction (`aggregation.py` draft)

Recall bin state rule: 
- If the bin is not fully observed or not fully known → **unknown**.
- Otherwise, cloud fraction `> 0.5` → **cloudy**; `< 0.5` → **clear**; `== 0.5` → **unknown** (tie).

In [3]:
def aggregate_to_bins(state, interval="20s", step=4):
    """Aggregate cloud state to time bins.

    Parameters
    ----------
    state : pd.Series
        Series of cloud state values (-1, 0, 1).
    interval : str, optional
        Resampling interval, by default "20s".
    step : int, optional
        Step size in seconds, by default 4.

    Returns
    -------
    pd.DataFrame
        DataFrame with aggregated cloud state statistics.
    """
    #Resample the cloud state series into time bins and compute statistics for each bin
    kw = dict(origin="start_day", label="left", closed="left") #Just like before
    known  = state.isin([0,1]).astype(np.int8).resample(interval, **kw).sum() #Count of known cloud states (0 or 1) in each bin
    cloudy = state.eq(1).astype(np.int8).resample(interval, **kw).sum() #Count of cloudy states (1) in each bin
    obs    = pd.Series(1, index=state.index, dtype=np.int8).resample(interval, **kw).sum() #Total number of observations in each bin
    expected = int(pd.Timedelta(interval).total_seconds()/step)
    return pd.DataFrame({
        "n_observed": obs, "n_known": known, "n_cloudy": cloudy,
        "cloud_fraction": cloudy/known.replace(0, np.nan),
        "fully_observed": obs.eq(expected),
        "fully_known": known.eq(obs) & obs.gt(0)})

#Now a helper function for determining the binned cloud state based on fully observed and fully known bins
def bin_state(bins):
    ok = bins.fully_observed & bins.fully_known #Bins that are both fully observed and fully known
    s = pd.Series(-1, index=bins.index, dtype=np.int8) #Initialize all bins as unknown (-1)
    s[ok & bins.cloud_fraction.gt(0.5)] = 1 #Set bins with cloud fraction > 0.5 as cloudy
    s[ok & bins.cloud_fraction.lt(0.5)] = 0 #Set bins with cloud fraction < 0.5 as clear
    return s

bins20 = aggregate_to_bins(state4, "20s"); s20 = bin_state(bins20)
assert len(bins20) == 1_581_120 #Check that the number of 20-second bins matches expectation
print(s20.value_counts())  # expect 2,705 unknown

 0    813816
 1    764599
-1      2705
Name: count, dtype: int64


## 4. Target Construction

Recall the target definitions:

**Target A — `PointState`:** at the current bin `t`, is bin `t+h` still cloudy?  
*Answers: "will the detected cloudy state still be reported at t+h?"*

**Target B — `Persistence`:** at `t`, is every bin from `t+1` through `t+h` cloudy?  
*Answers: "does the cloudy state survive the whole window without a clear break?"*

In [4]:
#Functions for building point and persistence state targets for cloud state prediction

#_window_counts counts the number of True values in a sliding window of length h for a boolean array ind
def _window_counts(ind, h):
    cs = np.r_[0, np.cumsum(ind, dtype=np.int64)] #Cumulative sum with a leading 0 for window counting
    i = np.arange(len(ind)) #Index array for the original array
    return cs[np.minimum(i+h+1, len(ind))] - cs[i+1] #Compute the number of True values in each sliding window of length h

def build_point_state_target(s, h):
    """Build point state target for cloud state prediction.

    Parameters
    ----------
    s : pd.Series
        Series of cloud state values (-1, 0, 1).
    h : int
        Horizon for the target.

    Returns
    -------
    pd.DataFrame
        DataFrame with current state, target, and target validity.
    """
    cur = s.to_numpy(); fut = s.shift(-h).to_numpy() #Current state and future state arrays
    valid = (cur==1) & np.isin(fut, [0,1]) #Valid target if current state is cloudy and future state is known
    tgt = np.where(valid, fut, -1).astype(np.int8) #Set target to future state if valid, else -1
    return pd.DataFrame({"current_state":cur,"target":tgt,"target_valid":valid}, index=s.index)

def build_persistence_target(s, h):
    """Build persistence target for cloud state prediction.

    Parameters
    ----------
    s : pd.Series
        Series of cloud state values (-1, 0, 1).
    h : int
        Horizon for the target.

    Returns
    -------
    pd.DataFrame
        DataFrame with current state, target, and target validity.
    """

    v = s.to_numpy(); n = len(v) #Current state array and its length
    clear = _window_counts(v==0, h) #Number of clear observations in the sliding window
    cloudy = _window_counts(v==1, h) #Number of cloudy observations in the sliding window
    in_range = np.arange(n) + h < n #Check if the window is within the array bounds
    all_cloudy = in_range & (cloudy == h) #All observations in the window are cloudy
    any_clear = clear > 0 #At least one clear observation in the window
    valid = (v==1) & (any_clear | all_cloudy) #Valid target if current state is cloudy and either any clear or all cloudy in the window
    tgt = np.where(valid, np.where(any_clear, 0, 1), -1).astype(np.int8) #Set target based on window observations if valid, else -1
    return pd.DataFrame({"current_state":v,"target":tgt,"target_valid":valid}, index=s.index)

## 5. Class Balance & Exclusion Exploration

Recall for each target/horizon, the **naive majority baseline** is `max(p_fraction, 1 - p_fraction)`, that is:

- Near 0.50 → balanced target; predictors must do real work.
- Near 1.00 → trivially easy; predictors can appear strong with no information.


In [5]:
HORIZONS = {"1min":3, "5min":15, "20min":60, "1h":180} #Mapping of horizon labels to number of steps
rows=[]

#Loop over each target type and horizon, compute statistics for cloudy rows and target validity
for name,b in [("PointState",build_point_state_target),("Persistence",build_persistence_target)]:
    for hl,h in HORIZONS.items(): #Loop over each horizon and compute targets
        t = b(s20,h); cur = t.current_state==1 #Boolean array indicating cloudy rows
        v = t.target_valid; y = t.target[v] #Boolean array indicating valid targets and the corresponding target values
        rows.append(dict(target=name, horizon=hl, n_cloudy_rows=int(cur.sum()), 
            n_valid=int(v.sum()), n_dropped=int((cur&~v).sum()),
            n_dissolved=int((y==0).sum()), p_persisted=(y==1).mean(),
            baseline=max((y==1).mean(), (y==0).mean())))
        
summary = pd.DataFrame(rows); display(summary.round(4))

,target,horizon,n_cloudy_rows,n_valid,n_dropped,n_dissolved,p_persisted,baseline
0,PointState,1min,764599,764570,29,23375,0.9694,0.9694
1,PointState,5min,764599,764511,88,51072,0.9332,0.9332
2,PointState,20min,764599,764452,147,81806,0.8930,0.8930
3,PointState,1h,764599,764325,274,116997,0.8469,0.8469
4,Persistence,1min,764599,764570,29,26194,0.9657,0.9657
5,Persistence,5min,764599,764473,126,77093,0.8992,0.8992
6,Persistence,20min,764599,764267,332,158795,0.7922,0.7922
7,Persistence,1h,764599,763865,734,257741,0.6626,0.6626


## 6. Effective sample size

Dissolved rows come from run endings, so this count is the real sample size. 

In [6]:
runs = (s20 != s20.shift()).cumsum() #Identify contiguous runs of the same state
cloudy_runs = s20[s20==1].groupby(runs[s20==1]).size() #Sizes of contiguous cloudy runs
print("cloudy bin-runs:", len(cloudy_runs)) 
print((cloudy_runs*20/60).describe()) #Summary statistics of cloudy run durations in hours

cloudy bin-runs: 9883
count    9883.000000
mean       25.788357
std       146.746820
min         0.333333
25%         1.000000
50%         2.000000
75%         7.000000
max      5938.000000
dtype: float64
